# Do the EC methods make the same mistakes?


**The cohort.** The frozen Plasmodium EC test genes of 68.02 (152 genes, seed 20261008): native random forest, kNN, the training-majority call and the prevalence-weighted call, each already scored. A call is correct when its EC set equals the truth set -- the definition that reproduces the recorded counts (forest 59, kNN 40, majority 43, prevalence 35).

**What is measured.** For every pair: agreement of the calls, how many genes both get right, both wrong, or only one; McNemar's exact test on the discordant genes (is one better beyond chance?); and the correlation of their errors. **Limit stated first:** both models read the same source features, so shared errors partly reflect shared inputs; this says nothing about biological accuracy.

Built 2026-11-07.

In [1]:
import itertools, json, os
import numpy as np, pandas as pd
from scipy.stats import binomtest
R = os.path.join(REPO, 'results')
rf = pd.read_parquet(f'{R}/pf_functional_ec_forest_2026_10_08_v2/rows.parquet').set_index('entity')
kn = pd.read_parquet(f'{R}/pf_functional_ec_knn_2026_10_08_v2/rows.parquet').set_index('entity')
ct = pd.read_parquet(f'{R}/pf_functional_ec_controls_2026_10_08/control_rows.parquet').set_index('entity')
assert set(rf.index) == set(kn.index) == set(ct.index), 'not the same cohort'
S = lambda v: frozenset(json.loads(v)) if isinstance(v, str) else frozenset()
truth = ct['truth'].map(S)
calls = pd.DataFrame({'random forest': rf['prediction'].map(S), 'kNN': kn['prediction'].reindex(ct.index).map(S),
                      'majority': ct['majority'].map(S), 'prevalence': ct['prevalence_call'].map(S)}).reindex(ct.index)
rf_ = rf.reindex(ct.index)
right = calls.apply(lambda c: c == truth)
right.sum().rename('correct of 152')

random forest    59
kNN              40
majority         43
prevalence       35

In [2]:
rows = []
for a, b in itertools.combinations(calls.columns, 2):
    ra, rb = right[a], right[b]
    only_a, only_b = int((ra & ~rb).sum()), int((rb & ~ra).sum())
    p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else float('nan')
    ea, eb = (~ra).astype(float), (~rb).astype(float)
    phi = float(np.corrcoef(ea, eb)[0, 1]) if ea.std() and eb.std() else float('nan')
    rows.append(dict(method_a=a, method_b=b, same_call=round(float((calls[a] == calls[b]).mean()), 3),
                     both_right=int((ra & rb).sum()), both_wrong=int((~ra & ~rb).sum()),
                     only_a_right=only_a, only_b_right=only_b, mcnemar_p=round(float(p), 4),
                     error_correlation=round(phi, 3)))
pairs = pd.DataFrame(rows)
pairs.to_csv(os.path.join(OUT, 'pairs.tsv'), sep='\t', index=False)
pairs

,method_a,method_b,same_call,both_right,both_wrong,only_a_right,only_b_right,mcnemar_p,error_correlation
0,random forest,kNN,0.618,33,86,26,7,0.0013,0.536
1,random forest,majority,0.487,24,74,35,19,0.0402,0.219
2,random forest,prevalence,0.303,17,75,42,18,0.0027,0.109
3,kNN,majority,0.401,13,82,27,30,0.7914,0.056
4,kNN,prevalence,0.283,12,89,28,23,0.5758,0.099
5,majority,prevalence,0.375,17,91,26,18,0.2912,0.246


### Would the two models together do better than either?

An upper bound, not a method: the share of genes at least one model gets right (an oracle that always picks the right one). If it is barely above the forest alone, kNN adds nothing the forest lacks.

In [3]:
either = int((right['random forest'] | right['kNN']).sum())
summary = pd.DataFrame([dict(genes=len(right), forest=int(right['random forest'].sum()), knn=int(right['kNN'].sum()),
                             either_oracle=either, both=int((right['random forest'] & right['kNN']).sum()))])
summary.to_csv(os.path.join(OUT, 'summary.tsv'), sep='\t', index=False)
summary

,genes,forest,knn,either_oracle,both
0,152,59,40,66,33
